In [1]:
import os
import time
import google.generativeai as genai
from dotenv import load_dotenv
import json
import tiktoken

C:\Users\10610\AppData\Local\Temp\ipykernel_18680\278477248.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
import os
import time

from dotenv import load_dotenv
import google.generativeai as genai

from services.tvmaze import search_shows

MAX_RECOMMENDATIONS = 5


def normalize_rating_value(value):
    if value is None:
        return ""
    text = str(value).strip().lower().replace("’", "'")
    if text in ["aime", "aimé"]:
        return "aimé"
    if text == "neutre":
        return "neutre"
    if text in ["n'aime pas", "n’aime pas", "deteste", "déteste"]:
        return "n'aime pas"
    return ""


def build_user_preferences(records):
    records = records or []
    aime = []
    neutre = []
    deteste = []

    for r in records:
        if not isinstance(r, dict):
            continue
        name = r.get("name_serie")
        if not name:
            continue
        value = normalize_rating_value(r.get("rating_value"))
        if value == "aimé":
            aime.append(name)
        elif value == "neutre":
            neutre.append(name)
        elif value == "n'aime pas":
            deteste.append(name)

    parts = []
    if aime:
        parts.append(f"Aime: {', '.join(aime)}")
    if neutre:
        parts.append(f"Neutre: {', '.join(neutre[:3])}")
    if deteste:
        parts.append(f"N'aime pas: {', '.join(deteste)}")
    return " ; ".join(parts)


def detect_intent(question):
    text = (question or "").lower()
    movie_words = ["电影", "電影", "movie", "film", "films", "cinema", "cinéma"]
    for word in movie_words:
        if word in text:
            return "movie"
    return "series"


def build_prompt(pref_text, question):
    if not pref_text:
        pref_text = "Aucune préférence enregistrée."
    question = (question or "").strip() or "Propose-moi des séries populaires."

    return (
        "Donne au maximum 5 recommendations de series TV.\n"
        "Format obligatoire: une ligne par recommandation, comme ceci:\n"
        "Nom de la serie - raison courte\n\n"
        f"Préférences utilisateur: {pref_text}\n"
        f"Question: {question}\n"
    )


def parse_lines(raw_text):
    candidates = []
    if not raw_text:
        return candidates

    lines = raw_text.splitlines()
    for line in lines:
        text = line.strip().lstrip("-•0123456789. ").strip()
        if not text:
            continue
        if " - " in text:
            left, right = text.split(" - ", 1)
            title = left.strip()
            reason = right.strip()
        else:
            title = text.strip()
            reason = "Recommandé selon votre demande."

        if title:
            candidates.append({"title": title, "reason": reason})

        if len(candidates) >= MAX_RECOMMENDATIONS:
            break

    return candidates


def pick_show(query, found):
    if not found:
        return None
    query_lower = query.lower().strip()
    for show in found:
        if (show.get("name") or "").lower().strip() == query_lower:
            return show
    return found[0]


def build_items(candidates):
    items = []
    for c in candidates:
        title = c.get("title", "").strip()
        reason = c.get("reason", "").strip() or "Recommandé selon votre demande."
        if not title:
            continue

        found = search_shows(title)
        show = pick_show(title, found)
        if not show:
            continue

        items.append(
            {
                "name": show.get("name") or title,
                "image": show.get("image"),
                "rating": show.get("rating"),
                "url": show.get("url"),
                "genres": show.get("genres", []),
                "reason": reason,
            }
        )

        if len(items) >= MAX_RECOMMENDATIONS:
            break

    return items


class GeminiSDK:
    def __init__(self):
        load_dotenv(override=True)
        model_name = os.getenv("GEMINI_MODEL", "gemini-flash-latest")
        api_key = (os.getenv("GEMINI_API_KEY") or "").strip().strip('"').strip("'")

        self.enabled = bool(api_key)
        self.model = None
        if self.enabled:
            genai.configure(api_key=api_key)
            self.model = genai.GenerativeModel(model_name)

    def generate(self, prompt):
        if not self.enabled or self.model is None:
            return "Le service IA n'est pas configuré (GEMINI_API_KEY manquante)."

        for attempt in range(3):
            try:
                t0 = time.time()
                response = self.model.generate_content(prompt)
                t1 = time.time()
                print(f"[TIME] Gemini generate_content: {t1 - t0:.3f}s")
                return response.text
            except Exception as exc:
                err = str(exc).lower()
                if "429" in err or "rate limit" in err or "504" in err:
                    print(f"[WARN] appel Gemini échoué ({attempt + 1}/3) : {err}")
                    time.sleep(2 * (attempt + 1))
                    continue
                if "401" in err or "403" in err or "api_key_invalid" in err:
                    return "Clé API Gemini invalide ou expirée."
                return f"Erreur Gemini: {exc}"

        return "Service temporairement indisponible, veuillez réessayer plus tard."


sdk = GeminiSDK()


def recommend_from_records(records, user_question):
    intent = detect_intent(user_question)
    if intent == "movie":
        return {
            "items": [],
            "message": "Actuellement, la recommandation est basée sur TVMaze (séries TV uniquement).",
        }

    pref_text = build_user_preferences(records)
    prompt = build_prompt(pref_text, user_question)
    raw_text = sdk.generate(prompt)

    if raw_text.startswith("Erreur Gemini") or raw_text.startswith("Clé API") or "manquante" in raw_text:
        return {"items": [], "message": raw_text}

    candidates = parse_lines(raw_text)
    items = build_items(candidates)

    if not items:
        return {
            "items": [],
            "message": "Aucune série TV trouvée dans TVMaze pour cette demande.",
        }

    message = ""
    if len(items) < 3:
        message = "Peu de résultats trouvés dans TVMaze. Essayez un style ou un genre plus précis."

    return {"items": items, "message": message}


def recommendation_user(records, question_utilisateur):
    return recommend_from_records(records, question_utilisateur)

In [ ]:
# 测试
import json

records_demo = [
    {"id_user": 1, "name_serie": "Breaking Bad", "rating_value": "aimé"},
    {"id_user": 1, "name_serie": "The Office", "rating_value": "neutre"},
    {"id_user": 1, "name_serie": "Riverdale", "rating_value": "n’aime pas"},
]

result = recommendation_user(records_demo, "Je veux une serie type thriller psychologique.")
print(json.dumps(result, ensure_ascii=False, indent=2))

[INFO] token_count=75
[TIME] Gemini generate_content: 106.218s
[TIME] total recommendation_user: 106.218s
Basé sur vos préférences (vous appréciez les chefs-d'œuvre narratifs profonds et les drames classiques, mais vous n'aimez pas les blockbusters d'action répétitifs ou purement commerciaux), voici cinq recommandations de films mondialement acclamés qui devraient vous plaire :

1. **《辛德勒的名单》(La Liste de Schindler)**
   - **Pourquoi :** Tout comme *Les Évadés*, c'est un film d'une immense puissance émotionnelle qui traite de la résilience humaine et de la moralité dans des moments sombres. C'est un chef-d'œuvre incontesté du cinéma.

2. **《阿甘正传》(Forrest Gump)**
   - **Pourquoi :** Un pilier du cinéma populaire qui, contrairement aux films d'action que vous n'aimez pas, mise tout sur l'écriture, le développement des personnages et une narration riche traversant l'histoire américaine.

3. **《盗梦空间》(Inception)**
   - **Pourquoi :** Si vous cherchez un film "populaire" mais intelligent. Con